# 09 — Test and run the metadata API

Run from the `notebooks/` folder, after notebook 08. Needs `fastapi` and `httpx` (`pip install -r requirements.txt`).

This notebook (1) checks that the database is usable, (2) exercises every endpoint in-process with FastAPI's test client (no network port needed), (3) shows how to start the real server and call it, and (4) exports the OpenAPI spec.

The service itself is in `../api/main.py`; it only reads `output/api/metadata.db`.

## 1. Locate the database and the app

In [3]:
from pathlib import Path
import json, os, sys, sqlite3
import pandas as pd
from IPython.display import display

ROOT = next((p for p in (Path.cwd(), Path.cwd().parent) if (p / 'api/main.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('api/main.py not found. Run from the project root or the notebooks folder.')
DB_PATH = ROOT / 'output/api/metadata.db'
if not DB_PATH.is_file():
    raise FileNotFoundError('Run notebook 08 first.')

os.environ['METADATA_DB'] = str(DB_PATH)
os.environ.pop('METADATA_API_KEY', None)
sys.path.insert(0, str(ROOT))
for name in [m for m in sys.modules if m == 'api' or m.startswith('api.')]:
    del sys.modules[name]                       # pick up edits to api/main.py on re-run
from api import main as api_main
from fastapi.testclient import TestClient
client = TestClient(api_main.app)
print('Database:', DB_PATH)

Database: /Users/georgyfurs/Documents/ICT_Year4/CPS/DDD/output/api/metadata.db


## 2. Database sanity checks

In [5]:
con = sqlite3.connect(f'{DB_PATH.as_uri()}?mode=ro', uri=True)
tables = [r[0] for r in con.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")]
print('Tables:', tables)
for t in ('documents', 'authors', 'keywords'):
    print(f'{t}: {con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]} rows')
print('Build info:', dict(con.execute('SELECT key, value FROM meta')))
con.close()

Tables: ['authors', 'documents', 'keywords', 'meta']
documents: 94 rows
authors: 76 rows
keywords: 98 rows
Build info: {'built_at': '2026-11-04T19:51:53+00:00', 'documents': '94', 'llm_models': 'Qwen/Qwen2.5-1.5B-Instruct', 'documents_with_llm_output': '8', 'keep_ungrounded': 'False', 'schema_version': '1'}


## 3. Exercise every endpoint

Each check asserts on the response, so a broken endpoint stops the notebook.

In [7]:
r = client.get('/health'); assert r.status_code == 200 and r.json()['status'] == 'ok'
print('health      ', r.json())

r = client.get('/v1/stats'); assert r.status_code == 200
stats = r.json()
print('stats       ', stats['documents'], 'documents; review status', stats['review_status'])
display(pd.DataFrame({'filled': stats['fields_filled']}))

r = client.get('/v1/documents', params={'limit': 5}); assert r.status_code == 200
page = r.json()
assert page['total'] == stats['documents'] and len(page['items']) <= 5
print('list        ', page['total'], 'total; first title:', page['items'][0]['title'])

first = page['items'][0]
r = client.get(f"/v1/documents/{first['document_id']}"); assert r.status_code == 200 and r.json() == first
print('by id       OK')
print(json.dumps(first, indent=2, ensure_ascii=False)[:1400])

health       {'status': 'ok', 'documents': 94, 'built_at': '2026-11-04T19:51:53+00:00'}
stats        94 documents; review status {'needs_review': 86, 'ok': 8}


,filled
title,38
authors,41
doi,33
publication_year,32
document_type,8
language,93
keywords,31


list         94 total; first title: Adaptive Capacities for Diversified Flood Risk Management Strategies: Learning from Pilot Projects
by id       OK
{
  "document_id": "doc_f5396bf79444d9ce",
  "file_name": "001_adaptive-capacities-for-diversified-flood-risk-management-strategies-learning-from-pilot-projects.pdf",
  "publinova_url": null,
  "page_count": 26,
  "title": "Adaptive Capacities for Diversified Flood Risk Management Strategies: Learning from Pilot Projects",
  "authors": [
    "Flavia Simona Cosoveanu, Jean-Marie Buijs, Marloes Bakker",
    "Teun Terpstra"
  ],
  "doi": "10.3390/w11122643",
  "publication_year": 2019,
  "document_type": null,
  "language": "en",
  "keywords": [
    "adaptive capacities",
    "diversiﬁed ﬂood risk management strategies",
    "pilot project",
    "governance"
  ],
  "review_status": "needs_review",
  "review_flags": [
    "no_llm_output"
  ],
  "provenance": {
    "title": {
      "source": "pdf_embedded",
      "confidence": "medium"
    },


In [8]:
# filters, sorting, paging
total = client.get('/v1/documents', params={'limit': 200}).json()
items = total['items']

if any(d['doi'] for d in items):
    doi = next(d['doi'] for d in items if d['doi'])
    for variant in (doi, f'https://doi.org/{doi}', doi.upper()):
        r = client.get(f'/v1/documents/by-doi/{variant}')
        assert r.status_code == 200 and r.json()['doi'] == doi, variant
    assert client.get('/v1/documents', params={'doi': doi}).json()['total'] >= 1
    print('by DOI      OK (plain, URL and upper-case forms)')

years = sorted({d['publication_year'] for d in items if d['publication_year']})
if years:
    y = years[0]
    r = client.get('/v1/documents', params={'year': y}).json()
    assert r['total'] >= 1 and all(d['publication_year'] == y for d in r['items'])
    r = client.get('/v1/documents', params={'year_from': years[-1], 'sort': '-year', 'limit': 3}).json()
    assert all(d['publication_year'] >= years[-1] for d in r['items'])
    print('year filter OK')

for lang in ('en', 'nl'):
    r = client.get('/v1/documents', params={'language': lang}).json()
    assert all(d['language'] == lang for d in r['items'])
    print(f'language={lang}:', r['total'])

r = client.get('/v1/documents', params={'min_confidence': 'high'}).json()
assert all(all(d['provenance'][f]['confidence'] == 'high' for f in ('title', 'authors', 'publication_year')) for d in r['items'])
print('min_confidence=high:', r['total'], 'documents')

r = client.get('/v1/documents', params={'review_status': 'needs_review'}).json()
print('needs_review:', r['total'])

authors = client.get('/v1/authors', params={'limit': 5}).json()
keywords = client.get('/v1/keywords', params={'limit': 5}).json()
print('top authors :', authors)
print('top keywords:', keywords)
if authors:
    r = client.get('/v1/documents', params={'author': authors[0]['value'].split()[-1]}).json()
    assert r['total'] >= 1
    print('author filter OK')
if keywords:
    r = client.get('/v1/documents', params={'keyword': keywords[0]['value'].upper()}).json()
    assert r['total'] >= 1
    print('keyword filter OK (case-insensitive)')

# paging does not repeat or skip documents
seen = []
for off in range(0, total['total'], 7):
    seen += [d['document_id'] for d in client.get('/v1/documents', params={'limit': 7, 'offset': off}).json()['items']]
assert len(seen) == len(set(seen)) == total['total']
print('paging      OK')

# LIKE wildcards in user input are treated literally
assert client.get('/v1/documents', params={'q': '%'}).json()['total'] == 0 or all('%' in (d['title'] or '') for d in client.get('/v1/documents', params={'q': '%'}).json()['items'])
print('wildcards   OK')

by DOI      OK (plain, URL and upper-case forms)
year filter OK
language=en: 50
language=nl: 43
min_confidence=high: 3 documents
needs_review: 86
top authors : [{'value': 'Vincent Bax', 'count': 5}, {'value': 'Bert Coleman', 'count': 4}, {'value': 'Anne Muilenburg', 'count': 3}, {'value': 'Christos Latsos', 'count': 2}, {'value': 'Ingrid Snijders', 'count': 2}]
top keywords: [{'value': 'Cryptophyceae', 'count': 2}, {'value': 'Microalgae production', 'count': 2}, {'value': 'Photobioreactors', 'count': 2}, {'value': 'relationship quality', 'count': 2}, {'value': 'science-and-literacy programs', 'count': 2}]
author filter OK
keyword filter OK (case-insensitive)
paging      OK
wildcards   OK


In [9]:
# error handling and API key
assert client.get('/v1/documents/does-not-exist').status_code == 404
assert client.get('/v1/documents/by-doi/10.0000/nothing').status_code == 404
assert client.get('/v1/documents', params={'limit': 1000}).status_code == 422
assert client.get('/v1/documents', params={'language': 'fr'}).status_code == 422
print('errors      OK (404 / 422)')

api_main.API_KEY = 'secret'
try:
    assert client.get('/v1/documents').status_code == 401
    assert client.get('/v1/documents', headers={'X-API-Key': 'wrong'}).status_code == 401
    assert client.get('/v1/documents', headers={'X-API-Key': 'secret'}).status_code == 200
    assert client.get('/health').status_code == 200          # health stays open
finally:
    api_main.API_KEY = None
print('api key     OK')
print('\nALL API CHECKS PASSED')

errors      OK (404 / 422)
api key     OK

ALL API CHECKS PASSED


## 4. Run the real server

From the **project root** (the folder that contains `api/`):

```bash
pip install -r requirements.txt
uvicorn api.main:app --host 127.0.0.1 --port 8000
```

Then open `http://127.0.0.1:8000/docs` for the interactive documentation. Example calls:

```bash
curl "http://127.0.0.1:8000/v1/documents?q=flood&language=en&min_confidence=medium&limit=10"
curl "http://127.0.0.1:8000/v1/documents/by-doi/10.3390/w11122643"
curl "http://127.0.0.1:8000/v1/stats"
```

```python
import requests
r = requests.get("http://127.0.0.1:8000/v1/documents", params={"year_from": 2020, "sort": "-year"})
for d in r.json()["items"]:
    print(d["publication_year"], d["title"], d["provenance"]["title"]["confidence"])
```

**Settings (environment variables):**


METADATA_DB - Use another database file. 
METADATA_API_KEY Require X-API-Key on all /v1 routes (/health stays open). 
CORS_ORIGINS Comma-separated browser origins allowed to call the API. 

**Updating the data:** rerun notebook 07 (new documents), then notebook 08. The database file is replaced atomically, so no restart is needed; each request opens the file fresh.

**Deploying:** a `Dockerfile` is included (`docker build -t metadata-api . && docker run -p 8000:8000 metadata-api`). Put it behind HTTPS (reverse proxy) and set `METADATA_API_KEY` if it is reachable from outside your network. The API serves metadata only; it does not expose PDFs or extracted text.

## 5. Export the OpenAPI spec

In [22]:
spec = client.get('/openapi.json').json()
(ROOT / 'output/api/openapi.json').write_text(json.dumps(spec, indent=2), encoding='utf-8')
print('Saved output/api/openapi.json')
display(pd.DataFrame([{'method': m.upper(), 'path': p, 'summary': op.get('summary')}
                      for p, ops in spec['paths'].items() for m, op in ops.items()]))

Saved output/api/openapi.json


,method,path,summary
0,GET,/health,Health
1,GET,/v1/documents,List Documents
2,GET,/v1/documents/by-doi/{doi},Get Document By Doi
3,GET,/v1/documents/{document_id},Get Document
4,GET,/v1/authors,List Authors
5,GET,/v1/keywords,List Keywords
6,GET,/v1/stats,Stats
